# TIR-ROI → Respiration — End-to-End Orchestration Notebook

This notebook implements the architecture specified in `notebooks/TirROI_Resp_Guide.md` by **reusing the existing, verified `code/` package**. It never re-implements datasets, ROI geometry, landmark parsing, models, losses or metrics — every computation below is a call into `code/`.

**Pipeline.** Thermal mouth+nose ROI (`tir`) → `MultiModalMAE` masked pre-training on all available selected data (Stage 2) → `MultiModalWaveformRegressor` subject-disjoint fine-tuning and validation (Stage 3) → graded evaluation (Tiers 1–2) and qualitative figures.

**How to run.** Execute top-to-bottom. The default is *smoke scale* (tiny caps, 1 epoch) so it finishes in a few minutes; cells marked `REAL RUN` are toggle stubs for full-scale training. The procedure of record is `code/plan/TirROI_Resp_plan.md`.

**Spec vs implementation.** Where the guide disagrees with the verified code (landmark indices, clip geometry, TIR channels, or split policy), the **code is ground truth**; every delta is listed in the final section. Stage 2 is explicitly pretrain-on-all (`pretrain_split: all`); subject-disjoint splitting is used only for Stage 3.

## Section 1 — Environment Setup & Accelerator Diagnostics

1.1 imports + path bootstrap, 1.2 hardware/accelerator audit, 1.3 determinism & seeding.

In [1]:
# 1.1 -- Imports, path bootstrap and shared notebook helpers.
# The notebook lives in <repo>/notebooks/; the pipeline lives in <repo>/code/.
import os
import sys
import json
import math
import random
import importlib
from pathlib import Path

import numpy as np
import torch
import matplotlib
import matplotlib.pyplot as plt

_here = Path.cwd()
REPO = _here.parent if (_here.parent / 'code').is_dir() else _here
CODE = REPO / 'code'
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
# The YAML configs use paths relative to code/ (../data/... , ../output/...).
os.chdir(CODE)

os.environ.setdefault('RAW_DATA_PATH', str(REPO / 'data' / 'raw' / 'BP4D'))
os.environ.setdefault('OUTPUT_DIR', str(REPO / 'output'))
os.environ.setdefault('DATA_PATH', str(REPO / 'data' / 'processed' / 'bp4d_canonical'))

NB_OUT = Path(os.environ['OUTPUT_DIR']) / 'notebook'
NB_OUT.mkdir(parents=True, exist_ok=True)

print('repo    :', REPO)
print('code    :', CODE)
print('raw data:', os.environ['RAW_DATA_PATH'])
print('out     :', NB_OUT)

repo    : /mnt/f/Studium/KISMed/github/MA-project-KISMED
code    : /mnt/f/Studium/KISMed/github/MA-project-KISMED/code
raw data: /mnt/f/Studium/KISMed/github/MA-project-KISMED/data/raw/BP4D
out     : /mnt/f/Studium/KISMed/github/MA-project-KISMED/output/notebook


In [2]:
# 1.1b -- Config loader.
# `run_pretrain.get_args()` / `run_waveform.get_args()` read `sys.argv`, so we
# temporarily swap it with a `-c <config>` command line. This guarantees the
# notebook and the shipped YAML configs can never drift apart.
def load_args(module_name, config, overrides=None):
    mod = importlib.import_module(module_name)
    cfg = Path(config)
    if not cfg.is_absolute():
        cfg = CODE / cfg
    argv = [module_name.split('.')[-1] + '.py', '-c', str(cfg)]
    for key, val in (overrides or {}).items():
        argv += ['--' + key, str(val)]
    saved = sys.argv
    sys.argv = argv
    try:
        return mod.get_args()
    finally:
        sys.argv = saved


def override(args, **kwargs):
    for key, val in kwargs.items():
        setattr(args, key, val)
    return args


print('helpers ready')

helpers ready


In [3]:
# 1.2 -- Hardware & accelerator diagnostics.
print('torch          :', torch.__version__)
print('cuda available :', torch.cuda.is_available())
print('device count   :', torch.cuda.device_count())
if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print('gpu            :', prop.name)
    print('compute cap    :', str(prop.major) + '.' + str(prop.minor))
    print('vram (GB)      :', round(prop.total_memory / 1024 ** 3, 2))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_OK = torch.cuda.is_available()
print('device         :', DEVICE, '| AMP usable:', AMP_OK)

torch          : 2.11.0+cu128
cuda available : True
device count   : 1
gpu            : NVIDIA GeForce RTX 5060 Ti
compute cap    : 12.0
vram (GB)      : 15.93
device         : cuda | AMP usable: True


In [4]:
# 1.3 -- Determinism & seed calibration.
# SPEC-VS-CODE: the repo helper `runners._common.init_env` sets
# `cudnn.benchmark=True` and never touches `cudnn.deterministic`. The guide asks
# for the opposite, so we seed explicitly and force the backend flags here.
SEED = 0


def seed_all(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_all()
print('seed', SEED, '| cudnn.deterministic', torch.backends.cudnn.deterministic,
      '| cudnn.benchmark', torch.backends.cudnn.benchmark)

seed 0 | cudnn.deterministic True | cudnn.benchmark False


## Section 2 — Global Configuration & Geometry Contracts

2.1 file-system contracts (the **raw** BP4D tree, not the canonical layout), 2.2 the 12-point thermal landmark contract, 2.3 the temporal/spatial geometry loaded from the shipped Stage-2 config.

In [5]:
# 2.1 -- File-system & path contracts.
# SPEC-VS-CODE: the guide writes `/BP4D+/Thermal/{Subject}/{Subject}_{Task}.wmv`.
# The real tree is `<raw_root>/Thermal/<S>/<T>.wmv` (the `<S>_<T>` form does not
# exist on disk); the dataset's resolver accepts both.
from data.tir_resp_dataset import default_raw_root, discover_sessions

RAW_ROOT = default_raw_root()
print('raw root :', RAW_ROOT)
for sub, pat in (('Thermal', '*.wmv'),
                 ('IRFeatures', '*.txt'),
                 ('Physiology', 'Resp_Volts.txt')):
    d = Path(RAW_ROOT) / sub
    files = list(d.rglob(pat)) if d.is_dir() else []
    print('  ' + sub.ljust(12), 'exists=' + str(d.is_dir()).ljust(5), pat, 'count=', len(files))

sessions = discover_sessions(RAW_ROOT)
print('sessions (thermal + IRFeatures):', len(sessions))
print('subjects :', sorted({s['subject'] for s in sessions}))
print('tasks    :', sorted({s['task'] for s in sessions}, key=lambda t: int(t[1:])))

raw root : /mnt/f/Studium/KISMed/github/MA-project-KISMED/data/raw/BP4D
  Thermal      exists=True  *.wmv count= 40
  IRFeatures   exists=True  *.txt count= 40
  Physiology   exists=True  Resp_Volts.txt count= 40
sessions (thermal + IRFeatures): 40
subjects : ['F001', 'F002', 'F003', 'F004']
tasks    : ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10']


In [6]:
# 2.2 -- Thermal landmark geometry contract (12 perinasal / lower-facial points).
# SPEC-VS-CODE: the guide lists 1-indexed [9,10,11,12,13,14,19,20,21,22,23,24].
# The verified code uses the set below (points 14/19 are brow/eye-lid, not
# perinasal; 25/26 are the lip centres) -- validated against decoded frames.
from data.tir_resp_dataset import (TARGET_LANDMARKS, TARGET_LANDMARK_IDX,
                                   ROI_LANDMARK_PRESETS, resolve_roi_landmarks)

print('1-indexed labels  :', TARGET_LANDMARKS)
print('0-indexed indices :', TARGET_LANDMARK_IDX)
for name, pts in ROI_LANDMARK_PRESETS.items():
    print('  preset ' + name.ljust(16), pts)
print('resolved default  :', resolve_roi_landmarks(''))

1-indexed labels  : (9, 10, 11, 12, 13, 20, 21, 22, 23, 24, 25, 26)
0-indexed indices : (8, 9, 10, 11, 12, 19, 20, 21, 22, 23, 24, 25)
  preset nose_mouth       (9, 10, 11, 12, 13, 20, 21, 22, 23, 24, 25, 26)
  preset nostrils         (10, 21)
  preset nostril_mouth    (10, 11, 21, 22)
  preset nose_tip         (9, 10, 20, 21)
resolved default  : (9, 10, 11, 12, 13, 20, 21, 22, 23, 24, 25, 26)


In [7]:
# 2.3 -- Temporal & spatial hyper-parameters, loaded from the shipped Stage-2 config.
PRETRAIN_CFG = 'configs/pretrain/stage2_local_tir_roi_resp.yaml'
FINETUNE_CFG = 'configs/finetune/resp_tir_roi_local.yaml'
p_args = load_args('runners.run_pretrain', PRETRAIN_CFG)

print('data_set        :', p_args.data_set)
print('streams         :', p_args.streams)
print('clip_duration   :', p_args.clip_duration, 's')
print('clip_stride     :', p_args.clip_stride, 's')
print('fps             :', p_args.fps)
print('temporal_stride :', p_args.temporal_stride)
print('tubelet         :', p_args.tubelet)
print('input_size      :', p_args.input_size)
print('fs              :', p_args.fs)
print('sig_kernel      :', p_args.sig_kernel)
print('roi_padding     :', p_args.roi_padding)
print('mask_ratio_tir  :', p_args.mask_ratio_tir)
print('mask_ratio_resp :', p_args.mask_ratio_resp)
print('target_norm     :', p_args.target_norm)
print('spectral_weight :', p_args.spectral_weight)

tubelet_t = int(str(p_args.tubelet).split(',')[0])
num_frames = int(round(p_args.clip_duration * p_args.fps / p_args.temporal_stride))
num_frames -= num_frames % tubelet_t
seq_len = int(round(num_frames * p_args.temporal_stride / p_args.fps * p_args.fs))
sig_kernel = int(p_args.sig_kernel)
grid_t = num_frames // tubelet_t
n_signal = seq_len // sig_kernel
print()
print('num_frames (T)  :', num_frames)
print('seq_len (L)     :', seq_len)
print('grid_t          :', grid_t)
print('n_signal        :', n_signal)
print('token seconds   :', tubelet_t * p_args.temporal_stride / p_args.fps,
      '== sig_kernel/fs', sig_kernel / p_args.fs)
print('alignment ok    :', grid_t == n_signal)

data_set        : tir_roi
streams         : tir,resp
clip_duration   : 8.0 s
clip_stride     : 4.0 s
fps             : 25.0
temporal_stride : 2
tubelet         : 2,16,16
input_size      : 64
fs              : 100.0
sig_kernel      : 16
roi_padding     : 0.2
mask_ratio_tir  : 0.9
mask_ratio_resp : 0.5
target_norm     : clip
spectral_weight : 0.0

num_frames (T)  : 100
seq_len (L)     : 800
grid_t          : 50
n_signal        : 50
token seconds   : 0.16 == sig_kernel/fs 0.16
alignment ok    : True


## Section 3 — Data Ingestion & Thermal ROI Pre-Processing

- **3.1** Landmark parsing + the `(0,0)` missing-data sentinel: IR features are parsed into `(frames, 28, 2)` landmark coordinates. Frames whose landmarks contain `(0,0)` are flagged as missing and excluded from reliable ROI processing.
- **3.2** The clip-static ROI box is computed once per clip from the selected thermal landmarks and shared across all frames, using the configured padding.
- **3.3** Respiration loader + quality guards: raw voltages are read from the physiology recording and sliced to the clip; zero-variance and minimum-spread guards reject degenerate windows that would poison the loss.
- **3.4** Normalisation contract: Stage 2 uses raw respiration voltages, which the model z-scores per clip; Stage 3 applies dataset-level z-score normalisation. Thermal frames are scaled to `[0,1]` inside the dataset, avoiding duplicate normalisation.


In [10]:
# 3.1 -- Landmark parsing + the (0,0) missing-data sentinel.
from data.tir_resp_dataset import parse_ir_features, missing_frame_mask

s0 = sessions[0]
ir = parse_ir_features(s0['ir_file'])
bad = missing_frame_mask(ir)
print('session         :', s0['session'])
print('ir array        :', ir.shape, ir.dtype)
print('sentinel frames :', int(bad.sum()), '/', len(bad))
print('landmark px     : x', float(np.nanmin(ir[:, :, 0])), '-', float(np.nanmax(ir[:, :, 0])),
      '| y', float(np.nanmin(ir[:, :, 1])), '-', float(np.nanmax(ir[:, :, 1])))

session         : F001_T1
ir array        : (1612, 28, 2) float32
sentinel frames : 0 / 1612
landmark px     : x 139.05511474609375 - 299.8360595703125 | y 234.16908264160156 - 393.7378845214844


In [11]:
# 3.2 -- Clip-static ROI box (Strategy 2): ONE box per clip, shared by all frames.
from data.tir_resp_dataset import BP4DPlusTIRRespDataset, roi_box_from_landmarks

base = BP4DPlusTIRRespDataset(raw_root=RAW_ROOT, subjects=[s0['subject']],
                              tasks=[s0['task']], clip_seconds=8.0,
                              input_size=p_args.input_size,
                              roi_padding=p_args.roi_padding,
                              allow_empty=True)
print(base.describe())
box = base.clip_roi_box(0)
print('clip 0 ROI box (x0,x1,y0,y1):', box,
      '->', box[1] - box[0], 'x', box[3] - box[2], 'px')
print('roi_padding used :', p_args.roi_padding, '(guide says 20% margin)')
print('landmarks in box :', base.target_idx)

raw_root        : /mnt/f/Studium/KISMed/github/MA-project-KISMED/data/raw/BP4D
sessions        : 1 used / 1 found / 0 skipped
clips           : 8 (0 window(s) dropped for a (0,0) sentinel line, 0 for a (near-)constant respiration window at min_signal_spread=0 V)
clip            : 8 s = 200 frames @ 25 fps, stride 200 frames; resp 8000 samples @ 1000 Hz
roi             : landmarks [9, 10, 11, 12, 13, 20, 21, 22, 23, 24, 25, 26] padding 0.2 -> 64x64
norm            : clip
task selection  : task_set "(none)" + explicit tasks T1 -> T1   [groups: (none)]
measured fps    : F001_T1=25.0000
clip 0 ROI box (x0,x1,y0,y1): (184, 269, 269, 372) -> 85 x 103 px
roi_padding used : 0.2 (guide says 20% margin)
landmarks in box : [ 8  9 10 11 12 19 20 21 22 23 24 25]


In [14]:
# 3.3 -- Respiration loader + quality guards.
resp_raw = base.respiration_clip(0, normalized=False)
full = np.loadtxt(s0['resp_file'])
print('clip (raw volts) :', resp_raw.shape,
      '| min', round(float(resp_raw.min()), 3),
      'max', round(float(resp_raw.max()), 3),
      'std', round(float(resp_raw.std()), 4))
print('full file        :', full.shape, 'samples  => fs',
      round(full.shape[0] / (len(ir) / p_args.fps), 1), 'Hz')
print('zero-variance guard (std < 1e-6) :', bool(resp_raw.std() < 1e-6))
print('min_signal_spread                :', getattr(p_args, 'min_signal_spread', 0.0),
      '| window spread', round(float(resp_raw.max() - resp_raw.min()), 4), 'V')

clip (raw volts) : (8000,) | min -6.476 max 0.334 std 1.9726
full file        : (64597,) samples  => fs 1001.8 Hz
zero-variance guard (std < 1e-6) : False
min_signal_spread                : 0.01 | window spread 6.81 V


In [15]:
# 3.4 -- Normalisation contract.
# Stage 2: the dataset returns RAW volts (norm='none') and the MODEL z-scores
# per clip via `target_norm: clip`. Stage 3: the dataset z-scores itself
# (`physio_norm: zscore`). Never normalise twice.
z = base.respiration_clip(0, normalized=True)
print('raw   : mean', round(float(resp_raw.mean()), 5), 'std', round(float(resp_raw.std()), 5))
print('zscore: mean', round(float(z.mean()), 5), 'std', round(float(z.std()), 5))
print('frames: pixels scaled to [0,1] by /255 inside the dataset')

raw   : mean -2.65846 std 1.97265
zscore: mean -0.0 std 1.0
frames: pixels scaled to [0,1] by /255 inside the dataset


## Section 4 — PyTorch Dataset & DataLoader Architecture

- **4.1 — Stage-2 dataset contract.** Build `TirRoiRespPretrainDataset` with `pretrain_split: all`, inspect `describe()`, and verify each item returns `tir [3,100,64,64]` (scaled to `[0,1]`) and `resp [1,800]` in **raw** volts. Stage 2 intentionally uses all selected subjects/sessions; the `max_entries` override below is only a smoke-test cap.
- **4.2 — DataLoader + batch integrity assertions.** Wrap the dataset with `make_data_loader` (`distributed=False`, `batch_size=2`) and assert the collated shapes are `(2,3,T,S,S)` and `(2,1,L)`, with a `check_loader_not_empty` guard.
- **4.3 — Subject-disjoint Stage-3 split.** Build train/val via `build_tir_roi_finetune_dataset`, confirm the 0.8 subject-disjoint split (or `--val_subject` LOSO) has no subject overlap, and sanity-check the z-scored waveform target.

In [16]:
# 4.1 -- Stage-2 dataset (TirRoiRespPretrainDataset) + item contract.
from data.tir_resp_dataset import build_tir_roi_pretrain_dataset

p_args = override(p_args, max_entries=4, num_workers=0)
ds2 = build_tir_roi_pretrain_dataset(p_args)
print(ds2.describe())
item = ds2[0]
print()
for key, val in item.items():
    print('  ' + key.ljust(6), tuple(val.shape), val.dtype)
print('tir range  :', float(item['tir'].min()), '-', float(item['tir'].max()))
print('resp range :', float(item['resp'].min()), '-', float(item['resp'].max()), '(raw volts)')

TirRoiRespPretrainDataset streams=['tir', 'resp']
  clips 4 from 1 session(s), ROI 64px padding 0.2
  geometry T=100 frames (8.0000 s), L=800 @ 100 Hz, temporal_stride=2
raw_root        : /mnt/f/Studium/KISMed/github/MA-project-KISMED/data/raw/BP4D
sessions        : 1 used / 40 found / 0 skipped
clips           : 4 (0 window(s) dropped for a (0,0) sentinel line, 0 for a (near-)constant respiration window at min_signal_spread=0.01 V)
clip            : 8 s = 200 frames @ 25 fps, stride 100 frames; resp 8000 samples @ 1000 Hz
roi             : landmarks [9, 10, 11, 12, 13, 20, 21, 22, 23, 24, 25, 26] padding 0.2 -> 64x64
norm            : none
task selection  : task_set "(none)" + explicit tasks none -> ALL tasks   [groups: high=T1,T5,T6,T9, low=T2,T3, moderate=T4,T7,T8,T10]
clips per level : high=4 clip(s)/1 session(s)
measured fps    : F001_T1=25.0000

  tir    (3, 100, 64, 64) torch.float32
  resp   (1, 800) torch.float32
tir range  : 0.0 - 1.0
resp range : -6.475830078125 - 0.33355700

In [17]:
# 4.2 -- DataLoader construction + batch shape integrity assertions.
from runners._common import make_data_loader, check_loader_not_empty

# `make_data_loader` reads `args.distributed`, which the runners set inside
# `init_env`; for a plain non-distributed loader we set it directly (no DDP).
p_args = override(p_args, distributed=False, batch_size=2, num_workers=0)
loader2 = make_data_loader(p_args, ds2, shuffle=False, drop_last=False)
check_loader_not_empty(loader2, 'stage2', p_args)
batch = next(iter(loader2))
T = ds2.num_frames
S = int(p_args.input_size)
L = ds2.seq_len
assert tuple(batch['tir'].shape) == (2, 3, T, S, S), batch['tir'].shape
assert tuple(batch['resp'].shape) == (2, 1, L), batch['resp'].shape
print('batch tir  :', tuple(batch['tir'].shape))
print('batch resp :', tuple(batch['resp'].shape))
print('assertions OK')

batch tir  : (2, 3, 100, 64, 64)
batch resp : (2, 1, 800)
assertions OK


In [18]:
# 4.3 -- Subject-disjoint train/val split.
# SPEC-VS-CODE: the guide asks for 70/15/15. The verified code does
# pretrain-on-all for Stage 2, and an 80/20 (or leave-one-subject-out, via
# --val_subject) subject-disjoint split for Stage 3.
from data.tir_resp_dataset import build_tir_roi_finetune_dataset

f_args = load_args('runners.run_waveform', FINETUNE_CFG,
                   overrides={'max_clips': 2, 'num_workers': 0})
ds_tr = build_tir_roi_finetune_dataset(True, False, f_args)
ds_va = build_tir_roi_finetune_dataset(False, False, f_args)
print('split_by  :', f_args.split_by, '| train_ratio', f_args.train_ratio)
print('train     :', len(ds_tr), 'clips | keys', ds_tr.split_keys,
      '| sessions', len(ds_tr.split_sessions))
print('val       :', len(ds_va), 'clips | keys', ds_va.split_keys,
      '| sessions', len(ds_va.split_sessions))
print('subject overlap:', sorted(set(ds_tr.split_keys) & set(ds_va.split_keys)))
tir_v, wave_v = ds_va[0]
print('val item tir   :', tuple(tir_v.shape), 'range', float(tir_v.min()), '-', float(tir_v.max()))
print('val item target:', tuple(wave_v.shape), 'mean', round(float(wave_v.mean()), 4),
      'std', round(float(wave_v.std()), 4))

[data] tir_roi_resp split_by=subject: train = ['F001', 'F002', 'F003'] (40 usable session(s), 4 subject(s))
[data] tir_roi_resp task_set "(none)" + explicit tasks none -> ALL tasks   [groups: high=T1,T5,T6,T9, low=T2,T3, moderate=T4,T7,T8,T10]
[data] tir_roi_resp split_by=subject: val = ['F004'] (40 usable session(s), 4 subject(s))
[data] tir_roi_resp task_set "(none)" + explicit tasks none -> ALL tasks   [groups: high=T1,T5,T6,T9, low=T2,T3, moderate=T4,T7,T8,T10]
split_by  : subject | train_ratio 0.8
train     : 58 clips | keys ['F001', 'F002', 'F003'] | sessions 29
val       : 20 clips | keys ['F004'] | sessions 10
subject overlap: []
val item tir   : (3, 100, 64, 64) range 0.0 - 1.0
val item target: (800,) mean -0.0 std 1.0006


## Section 5 — Model Architecture (`MultiModalMAE`)

5.1 patch stem + positional encoding, 5.2 joint spatio-temporal encoder, 5.3 the 1-D waveform head, 5.4 Stage-1 weight ingestion.

In [ ]:
# 5.1 -- Stage-2 model (MultiModalMAE): tubelet stem, sincos positions, encoder, heads.
from core.multimae import build_pretraining_model

model2 = build_pretraining_model(p_args)
n_par = sum(p.numel() for p in model2.parameters())
print('params           :', str(round(n_par / 1e6, 1)) + ' M')
for name in ('adapters', 'signal_adapters', 'positions', 'heads'):
    obj = getattr(model2, name, None)
    if obj is None:
        print(name.ljust(16), 'n/a')
    elif hasattr(obj, 'keys'):
        print(name.ljust(16), list(obj.keys()))
    else:
        print(name.ljust(16), type(obj).__name__)
print('enc blocks       :', len(model2.enc_blocks))
print('dec blocks       :', len(model2.dec_blocks))
print('tubelet          :', getattr(model2, 'tubelet', None))
try:
    pos = model2.positions['tir'].pos_embed
    print('pos embed (tir)  :', tuple(pos.shape))
except Exception as exc:
    print('pos embed probe failed:', exc)

In [ ]:
# 5.2 -- Forward pass: masked-reconstruction loss decomposition.
model2.eval()
with torch.no_grad():
    out = model2({'tir': batch['tir'].to(DEVICE), 'resp': batch['resp'].to(DEVICE)})
print('total loss      :', round(float(out['loss']), 4))
for stream, val in out['losses_mse'].items():
    print('  mse_' + stream.ljust(6), round(float(val), 4))
print('weighted contrib:', {k: round(float(v), 4) for k, v in out['losses'].items()})
print('spectral terms  :', out['losses_spectral'])
print('mask fractions  :', {k: round(float(v.float().mean()), 3) for k, v in out['masks'].items()})
for stream, pred in out['preds'].items():
    print('  pred_' + stream.ljust(6), tuple(pred.shape))

In [ ]:
# 5.3 -- Stage-1 encoder init.
# EXACT reuse of core.multimae.load_pretrained_encoder: a tir-only model inherits
# the VideoMAE Conv3d(3,768,(2,16,16)) tubelet verbatim (the patch-embed
# destination is resolved to the first available VISUAL adapter).
from core.multimae import load_pretrained_encoder
from models.pretrained import resolve_encoder_weights

model2_init = build_pretraining_model(p_args)
try:
    ckpt_path = resolve_encoder_weights('videomae:k400')
    report = load_pretrained_encoder(model2_init, ckpt_path)
    print('init report:', report)
except Exception as exc:
    print('Stage-1 checkpoint unavailable (offline?):', type(exc).__name__, exc)

In [ ]:
# 5.4 -- Stage-3 1-D waveform head: grid_t tokens -> output_len samples.
# SPEC-VS-CODE: the guide's 5.4 also asks for layer-wise LR decay (0.65).
# `utils/optim_factory.create_optimizer` SUPPORTS it via get_num_layer /
# get_layer_scale, but NO runner passes an assigner -- so it is not active in
# any shipped config (documented in the final table).
from core.waveform_model import build_waveform_model

model3 = build_waveform_model(f_args)
print('streams           :', model3.streams)
print('num_frames        :', model3.num_frames)
print('grid_t            :', model3.grid_t)
print('samples_per_token :', model3.samples_per_token)
print('output_len        :', model3.output_len)
print('sec/token         :', model3.sec_per_visual_token)
print('head              :', model3.waveform_head)

## Section 6 — Stage-2 Multimodal Masked Pre-Training

Stage 2 is self-supervised pre-training on all available selected subjects, sessions, and clips (`pretrain_split: all`) with no train/validation split. The smoke cell uses a temporary `max_entries` cap only for fast verification. 6.1 the spatial-temporal tubelet masker + physio masking, 6.2 the masked-MSE objective, 6.3 the pre-training loop.

In [ ]:
# 6.1 -- Masking engine: tubelet masking (visual) + scattered/span masking (physio).
model2.eval()
masks = model2.make_masks(batch['tir'].shape[0], DEVICE)
for stream, mask in masks.items():
    print(stream.ljust(6), 'tokens', str(mask.shape[1]).ljust(5),
          'masked_frac', round(float(mask.float().mean()), 3))

fig, axes = plt.subplots(len(masks), 1, figsize=(10, 1.1 * len(masks)), squeeze=False)
for ax, (stream, mask) in zip(axes[:, 0], masks.items()):
    ax.imshow(mask.cpu().numpy(), aspect='auto', cmap='gray_r', interpolation='nearest')
    ax.set_ylabel(stream)
    ax.set_yticks([])
axes[0, 0].set_title('masked tokens (black = masked)')
plt.tight_layout()
plt.show()

In [ ]:
# 6.2 -- Objective as shipped: per-modality MASKED MSE.
# SPEC-VS-CODE: the guide's 6.2 masked MSE matches. The STFT term is NOT part
# of Stage 2 (spectral_weight 0.0 in every configs/pretrain/*.yaml) -- it is a
# FULL-weight Stage-3 term on the final waveform.
weighted = sum(float(v) for v in out['losses'].values())
print('sum(weighted contribs):', round(weighted, 5))
print('out[loss]             :', round(float(out['loss']), 5))
print('reconciles            :', abs(weighted - float(out['loss'])) < 1e-4)
print('spectral enabled      :', bool(out['losses_spectral']))

In [ ]:
# 6.3 -- Stage-2 pre-training loop, SMOKE scale (1 epoch, warmup 0, capped entries).
# Production Stage 2 uses every selected subject/session (pretrain_split='all').
# `warmup_epochs 0` is REQUIRED for a 1-epoch smoke: utils.lr_sched.cosine_scheduler
# asserts len(schedule) == epochs * steps.
from engines import train_one_epoch_pretrain
from utils.lr_sched import cosine_scheduler
from utils.native_scaler import NativeScalerWithGradNormCount
from utils.optim_factory import create_optimizer
from utils.checkpoint import save_model
from runners._common import init_env

p_run = load_args('runners.run_pretrain', PRETRAIN_CFG,
                  overrides={'epochs': 1, 'warmup_epochs': 0, 'max_entries': 4,
                             'batch_size': 2, 'num_workers': 0,
                             'output_dir': str(NB_OUT / 'stage2_smoke')})
assert p_run.pretrain_split == 'all'
seed_all()
device = init_env(p_run)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ds2 = build_tir_roi_pretrain_dataset(p_run)
loader2 = make_data_loader(p_run, ds2, shuffle=True, drop_last=False)
check_loader_not_empty(loader2, 'stage2', p_run)

model2 = build_pretraining_model(p_run).to(device)
optimizer = create_optimizer(p_run, model2)
scaler = NativeScalerWithGradNormCount()
steps = len(loader2)
schedule = cosine_scheduler(p_run.lr, p_run.min_lr, p_run.epochs, steps,
                            warmup_epochs=p_run.warmup_epochs)
for epoch in range(p_run.epochs):
    stats = train_one_epoch_pretrain(model2, loader2, optimizer, device, epoch,
                                     loss_scaler=scaler, max_norm=p_run.clip_grad,
                                     lr_schedule_values=schedule,
                                     start_steps=epoch * steps)
    keep = ('loss', 'grad_norm', 'mse_tir', 'mse_resp')
    print('epoch', epoch, {k: round(float(v), 4) for k, v in stats.items() if k in keep})
save_model(p_run, p_run.epochs - 1, model2, model2, optimizer, scaler)
print('checkpoint written to', p_run.output_dir)

In [ ]:
# 6.4 -- REAL Stage-2 run (toggle).
# This run uses all available data; unlike the smoke cell, it has no entry cap.
# ~45 min locally (40 epochs, ~385 clips at batch 4) -- see code/plan/TirROI_Resp_plan.md P6.
# Uncomment to execute; kept explicit so Run All stays cheap.
# REAL = load_args('runners.run_pretrain', PRETRAIN_CFG)
# from runners.run_pretrain import main as run_pretrain_main
# run_pretrain_main(REAL)
print('Stage-2 real run:')
print('  python -u runners/run_pretrain.py -c ' + PRETRAIN_CFG)
print('  (pipe through `tee <out>/train.log`; -u is required or the log stays empty)')

## Section 7 — Stage-3 Supervised Fine-Tuning

Stage 3 is the only stage that creates a train/validation split: the default is subject-disjoint 80/20, with optional leave-one-subject-out validation via `val_subject`. 7.1 the composite loss, 7.2 Stage-2 encoder transfer + ROI contract, 7.3 the fine-tuning loop.

In [ ]:
# 7.1 -- Stage-3 composite loss + per-term decomposition.
# WaveformJointLoss = alpha*StdLoss + beta*PearsonLoss + gamma*MR-STFT.
# SPEC-VS-CODE: this MATCHES the guide's CompositeLoss (Pearson + std +
# log-magnitude spectral L1); the repo only differs by using StdLoss for the
# amplitude term instead of a raw L1.
from core import WaveformJointLoss

fft_sizes = tuple(int(x) for x in str(f_args.fft_sizes).split(','))
crit = WaveformJointLoss(f_args.alpha, f_args.beta, f_args.gamma, fft_sizes)
print(crit)
print('alpha(StdLoss) =', f_args.alpha, '| beta(Pearson) =', f_args.beta,
      '| gamma(MR-STFT) =', f_args.gamma, '| fft_sizes', fft_sizes)
with torch.no_grad():
    p = torch.randn(4, model3.output_len)
    t = torch.randn(4, model3.output_len)
    print('  std     :', round(float(crit.std(p, t)), 4))
    print('  pearson :', round(float(crit.pearson(p, t)), 4))
    print('  mr-stft :', round(float(crit.stft(p, t)), 4))
    print('  joint   :', round(float(crit(p, t)), 4))

In [ ]:
# 7.2 -- Transfer the Stage-2 encoder and verify the ROI contract.
# `check_stage2_roi_contract` compares the CHECKPOINT's roi_landmarks /
# roi_padding / roi_quantile / input_size with the Stage-3 config and raises
# SystemExit on a mismatch -- a ROI swap changes pixels but no tensor shape, so
# without this gate a mismatched run loads 150 tensors cleanly and silently
# fine-tunes on pixels the encoder never saw.
from core.waveform_model import load_stage2_encoder
from runners.run_waveform import check_stage2_roi_contract

model3 = build_waveform_model(f_args)
info = load_stage2_encoder(model3, f_args.finetune, target=f_args.target)
print('loaded        :', info['loaded'], '| skipped', info['skipped'],
      '| shape_mismatch', info['shape_mismatch'])
print('head source   :', info['head_source'], info['head_loaded'])
print('roi contract  :', check_stage2_roi_contract(f_args, info['stage2_args']))

ck = torch.load(f_args.finetune, map_location='cpu', weights_only=False)
transferred = torch.equal(model3.waveform_head.weight, ck['model']['heads.resp.weight'])
print('waveform_head.weight == ckpt heads.resp.weight :', transferred)

In [ ]:
# 7.3 -- Stage-3 fine-tuning loop, SMOKE scale.
from engines import train_one_epoch_waveform, evaluate_waveforms, predict_waveforms

f_run = load_args('runners.run_waveform', FINETUNE_CFG,
                  overrides={'epochs': 1, 'warmup_epochs': 0, 'max_clips': 2,
                             'batch_size': 2, 'num_workers': 0,
                             'output_dir': str(NB_OUT / 'stage3_smoke')})
seed_all()
device = init_env(f_run)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ds_tr = build_tir_roi_finetune_dataset(True, False, f_run)
ds_va = build_tir_roi_finetune_dataset(False, False, f_run)
ld_tr = make_data_loader(f_run, ds_tr, shuffle=True, drop_last=False)
ld_va = make_data_loader(f_run, ds_va, shuffle=False, drop_last=False)
check_loader_not_empty(ld_tr, 'stage3 train', f_run)

model3 = build_waveform_model(f_run).to(device)
info = load_stage2_encoder(model3, f_run.finetune, target=f_run.target)
check_stage2_roi_contract(f_run, info['stage2_args'])

fft_sizes = tuple(int(x) for x in str(f_run.fft_sizes).split(','))
crit = WaveformJointLoss(f_run.alpha, f_run.beta, f_run.gamma, fft_sizes)
optimizer = create_optimizer(f_run, model3)
scaler = NativeScalerWithGradNormCount()
steps = len(ld_tr)
schedule = cosine_scheduler(f_run.lr, f_run.min_lr, f_run.epochs, steps,
                            warmup_epochs=f_run.warmup_epochs)
best = None
for epoch in range(f_run.epochs):
    stats = train_one_epoch_waveform(model3, crit, ld_tr, optimizer, device, epoch,
                                     loss_scaler=scaler, max_norm=f_run.clip_grad,
                                     lr_schedule_values=schedule,
                                     start_steps=epoch * steps)
    metrics = evaluate_waveforms(ld_va, model3, device, fs=f_run.fs, band=f_run.eval_band)
    print('epoch', epoch, '| loss', round(float(stats['loss']), 4), '|', metrics)
    best = metrics.get('pearson', float('nan')) if best is None else max(best, metrics.get('pearson', float('nan')))
pred, target = predict_waveforms(ld_va, model3, device)
print('pred', pred.shape, '| target', target.shape, '| best pearson', round(float(best), 4))

In [ ]:
# 7.4 -- REAL Stage-3 run (toggle).
# ~2 h locally (40 epochs, 147 steps/epoch at batch 4) -- see code/plan/TirROI_Resp_plan.md
# 10.7. Uncomment to execute.
# REAL = load_args('runners.run_waveform', FINETUNE_CFG)
# from runners.run_waveform import main as run_waveform_main
# run_waveform_main(REAL)
print('Stage-3 real run:')
print('  python -u runners/run_waveform.py -c ' + FINETUNE_CFG)
print('the existing full run is at output/finetune/resp_tir_roi_local/')

## Section 8 — Model Evaluation & Signal Diagnostics

8.1 Tier-1 time-domain metrics, 8.2 Tier-2 spectral metrics + respiration rate, 8.3 task / distortion-group stratification.

In [ ]:
# 8.1 -- Tier-1 time-domain metrics (MAE / RMSE / Pearson).
from evaluation.metrics import time_domain_metrics, spectral_metrics

tier1 = time_domain_metrics(pred, target)
print('Tier-1 (macro over clips):')
for key, val in tier1.items():
    print('  ' + key.ljust(8), round(float(val), 4))

In [ ]:
# 8.2 -- Tier-2 spectral diagnostics + respiration rate (BPM).
from scipy.signal import welch

band = tuple(float(x) for x in str(f_run.eval_band).split(','))
tier2 = spectral_metrics(pred, target, fs=f_run.fs, band=band)
print('Tier-2 (band', band, 'Hz):')
for key, val in tier2.items():
    print('  ' + key.ljust(22), val)


def dominant_hz(sig, fs, band):
    x = np.asarray(sig, dtype=float).reshape(-1)
    f, pxx = welch(x, fs=fs, nperseg=int(min(len(x), 256)))
    sel = (f >= band[0]) & (f <= band[1])
    if not sel.any():
        return float('nan')
    return float(f[sel][int(np.argmax(pxx[sel]))])


gt_bpm = np.array([dominant_hz(row, f_run.fs, band) * 60.0 for row in target])
pr_bpm = np.array([dominant_hz(row, f_run.fs, band) * 60.0 for row in pred])
print('GT BPM mean   :', round(float(np.nanmean(gt_bpm)), 2))
print('pred BPM mean :', round(float(np.nanmean(pr_bpm)), 2))
print('BPM MAE       :', round(float(np.nanmean(np.abs(gt_bpm - pr_bpm))), 3))

In [ ]:
# 8.3 -- Task-wise / distortion-group stratification.
# `task_level` comes from the dataset's task-group resolver (configs carry
# low / moderate / high task groups). Row order of pred/target follows the
# SequentialSampler, i.e. dataset order, so it aligns with ds_va.entries.
entries = ds_va.entries
levels = np.array([(e.get('task_level') or ['?'])[0] for e in entries])
print('clip level distribution:', {lv: int((levels == lv).sum()) for lv in sorted(set(levels))})
for level in sorted(set(levels)):
    idx = np.where(levels == level)[0]
    if len(idx) == 0:
        continue
    mt = time_domain_metrics(pred[idx], target[idx])
    print(level.ljust(10), 'n=' + str(len(idx)).ljust(4),
          {k: round(float(v), 4) for k, v in mt.items()})

## Section 9 — Visualization & Qualitative Analysis

9.1 time-domain overlays, 9.2 log-PSD comparison, 9.3 distortion-group summary chart.

In [ ]:
# 9.1 -- Time-domain overlay of ground truth vs prediction.
from evaluation.report import plot_waveform_panel

png = NB_OUT / 'waveform_panel.png'
labels = [e['session'] for e in entries]
try:
    plot_waveform_panel(pred, target, fs=f_run.fs, out_png=str(png),
                        title='TIR-ROI to respiration (smoke)',
                        band=band, metrics=tier1, max_clips=4, sessions=labels)
    print('wrote', png)
except Exception as exc:
    print('plot_waveform_panel failed:', type(exc).__name__, exc)

In [ ]:
# 9.2 -- Log-PSD overlay.
# NEW plotting only (the repo has no PSD-overlay helper); the signal processing
# itself is scipy Welch, the same estimator used by evaluation.metrics.
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for k in range(int(min(3, len(pred)))):
    nperseg = int(min(target.shape[1], 256))
    f, pt = welch(target[k], fs=f_run.fs, nperseg=nperseg)
    _, pp = welch(pred[k], fs=f_run.fs, nperseg=nperseg)
    ax[0].semilogy(f, pt + 1e-12, label='clip ' + str(k))
    ax[1].semilogy(f, pp + 1e-12, label='clip ' + str(k))
ax[0].set_title('ground-truth PSD')
ax[1].set_title('prediction PSD')
for a in ax:
    a.set_xlabel('Hz')
    a.set_xlim(0.0, 1.0)
    a.grid(True, alpha=0.3)
    a.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
# 9.3 -- Per-distortion-group Pearson bar chart.
names, pears = [], []
for level in sorted(set(levels)):
    idx = np.where(levels == level)[0]
    names.append(level + ' (n=' + str(len(idx)) + ')')
    pears.append(float(time_domain_metrics(pred[idx], target[idx])['pearson']))
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(names, pears, color='#4c72b0')
ax.axhline(0.0, color='0.6', lw=0.8)
ax.set_ylabel('Pearson r')
ax.set_title('Task-group respiration fidelity')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## Verification & Spec-vs-Implementation

Run the repository's own self-tests, then read the table of deltas between the guide and the verified code.

In [ ]:
# Verification -- the repository's own self-tests (all should report PASS).
import subprocess

checks = ([sys.executable, '-m', 'core.multimae'],
          [sys.executable, 'data/tir_resp_dataset.py', '--check_views'],
          [sys.executable, '-m', 'evaluation.metrics'])
for cmd in checks:
    res = subprocess.run(cmd, cwd=str(CODE), capture_output=True, text=True)
    text = (res.stdout or '') + (res.stderr or '')
    lines = [ln for ln in text.strip().splitlines() if ln.strip()]
    print(('PASS' if res.returncode == 0 else 'FAIL'), '| exit', res.returncode,
          '|', ' '.join(cmd), '|', lines[-1] if lines else '')

### Spec vs implementation

| Item | `TirROI_Resp_Guide.md` | Verified `code/` (ground truth) |
| --- | --- | --- |
| Landmarks (1-indexed) | `[9,10,11,12,13,14,19,20,21,22,23,24]` | `(9,10,11,12,13,20,21,22,23,24,25,26)` — 14/19 are brow/eye-lid, 25/26 are lip centres; validated on decoded frames |
| Clip duration / frames | 8 s / 200 frames @ 25 fps | 8 s / `T = 100` (`temporal_stride: 2`) |
| Respiration samples | `N = 8000` @ 1000 Hz | `L = 800` @ 100 Hz (raw 1000 Hz resampled onto the model grid) |
| Crop size | 112 × 112 | 64 × 64 (`input_size`) |
| BBox margin | 20 % | `roi_padding: 0.2` (+40 % overall, each side by 0.2×extent) |
| TIR channels | 1 (grayscale) | 3 — the thermal stream is a false-colour rendering with a burned-in legend |
| Dataset contract | `tir_video (1,200,112,112)` + `resp_signal (8000,)` | Stage 2: `{'tir' [3,100,64,64], 'resp' [1,800]}`; Stage 3: tuple `(tir, waveform)` |
| Split | 70 / 15 / 15 subject-disjoint | Stage 2 uses all selected data (`pretrain_split: all`); Stage 3 subject-disjoint 0.8 (or `--val_subject` LOSO over the 4 local subjects) |
| Loss | `CompositeLoss` = Pearson + std + log-FFT L1 | `WaveformJointLoss` = `alpha*StdLoss + beta*PearsonLoss + gamma*MR-STFT` — **matches** |
| Stage-2 spectral term | implied by the loss | OFF in Stage 2 (`spectral_weight 0.0`); applied at full weight in Stage 3 |
| Grad-norm clip | 1.0 | 5.0 (measured-safe; the branch carries a `gamma = 1.0` MR-STFT term) |
| Layer-wise LR decay 0.65 | required (§5.4) | supported by `utils/optim_factory` but no runner wires an assigner → not active |
| Determinism | `cudnn.deterministic=True`, `benchmark=False` | `init_env` sets `benchmark=True`; the notebook overrides both after `init_env` |

**Note.** `code/plan/TirROI_Resp_plan.md` is the procedure of record for every number in the right-hand column; `code/README.md` documents the CLI surfaces.